# Week 2: ML Task Framing & Label Mechanics (ML-03)

**Track:** Applied Search Intelligence: Google Search Ranking & Discoverability  
**Lane:** Refresh / Content Opportunity Scoring  
**Author:** Altaf (ML Intern)  
**Dataset:** Starter Anonymized Release (`data/raw/content_refresh_anonymized.csv`)  
**Primary Metric:** Precision@50 (Target base rate: 54.2%)  

---

## 1. Problem Formulation & Task Type

In Week 1 (ML-02), we identified our operational goal: *Which aging search articles should an editorial team review and refresh first?*

### 1.1 Why Priority Ranking Instead of Raw Binary Classification?
A standard binary classifier outputs a discrete decision: $\hat{y} \in \{0, 1\}$ based on a static decision threshold (e.g., $p \ge 0.5$). In production, this causes two critical operational failures:
1. **Capacity Overload:** If 8,000 pages exceed $p = 0.5$, an editorial team capable of reviewing only 50 pages per week has no guidance on which 50 to tackle on Monday morning.
2. **Capacity Starvation:** In low-traffic seasonal periods, a strict cutoff might return 5 pages, leaving editorial staff under-utilized.

Therefore, we frame this as a **decision-support priority ranking task**. The model estimates the posterior probability of future organic click decline $\hat{p}_i = P(y_i = 1 \mid \mathbf{x}_i)$. Pages are then sorted in descending order of predicted risk to yield a prioritized work queue.

## 2. Temporal Window Alignment & Label Definition

A rigorous machine learning model requires an exact, non-overlapping temporal boundary separating feature observations from the target outcome.

```text
FEATURE OBSERVATION WINDOW (90 Days)            EVALUATION HORIZON (30 Days)
[================== T_-90 to T_0 ==================] | [====== T_0 to T_+30 ======]
  - Impressions, Clicks, GA4 Sessions                 - Forward organic clicks
  - Average SERP Position                             - Target: is_declining_label
  - Content Age, Staleness (Days Since Update)          (Click drop >= 15%)
  - Engagement & Scroll Telemetry
```

### 2.1 Formal Target Definition
$$\text{is\_declining\_label}_i = \begin{cases} 1 & \text{if } \frac{\text{Clicks}_{i, [T_0, T_{+30}]} - \text{Clicks}_{i, [T_{-30}, T_0]}}{\text{Clicks}_{i, [T_{-30}, T_0]} + \epsilon} \le -0.15 \\[6pt] 0 & \text{otherwise} \end{cases}$$

### 2.2 Base Rate Measurement
In the starter release of 30,000 anonymized pages across 32 clients:
- **Positive Label Rows (`is_declining_label = 1`):** 16,262
- **Negative Label Rows (`is_declining_label = 0`):** 13,738
- **Empirical Base Rate:** **54.2%**

> **Rubric Principle:** Any evaluation metric (such as Precision@50) must always be interpreted relative to this 54.2% majority-class base rate. A model predicting randomly achieves ~54% precision; our learned ranker must demonstrate substantial positive lift over this floor.

## 3. The Label Trap & Circularity Audit

One of the most dangerous failure modes in search intelligence ML is **target leakage through label-derived features**.

### 3.1 The `trend_direction` and `trend_pct` Trap
The dataset contains two pre-calculated trajectory columns: `trend_direction` (`down`, `flat`, `up`) and `trend_pct`. Because `is_declining_label` was calculated directly from this trajectory:
- Feeding `trend_direction` into training yields an artificial ROC AUC of **0.992** and Precision@50 of **1.000**.
- In live production at time $T_0$, the forward trend does not exist yet. A model reliant on `trend_direction` instantly collapses.
- **Strict Policy:** `trend_direction` and `trend_pct` are **banned from the feature matrix $\mathbf{X}$**.

### 3.2 The Product Flags Circularity Trap
Production systems often contain hand-written heuristic flags (`health_score`, `quick_win_flag`, `needs_review`).
- These flags are human decisions already made by existing heuristics.
- Training a model on `health_score` creates a circular system: the model learns to imitate the old rules rather than discovering genuine decay patterns in raw telemetry.
- **Strict Policy:** Product flags are benchmark baselines to beat, never model features.

## 4. Evaluation Metric Justification: Why Precision@50?

| Candidate Metric | Formula / Scope | Operational Fit for FlyRank | Verdict |
|---|---|---|---|
| **Accuracy** | $\frac{\text{TP} + \text{TN}}{N}$ | Misleading. A trivial dummy model predicting "decline" for all pages scores 54.2% accuracy without providing ranking value. | ❌ Rejected |
| **ROC AUC** | Area under ROC Curve | Measures global discrimination across all 30,000 items. Reviewers will never examine rank 15,000; ROC AUC weights errors in the bottom tail equally with errors at rank #1. | ⚠️ Secondary Diagnostic Only |
| **Average Precision (PR AUC)** | Area under PR Curve | Excellent for whole-curve precision under class imbalance, but still evaluates depths beyond human capacity. | ⚠️ Secondary Diagnostic Only |
| **Precision@50** | $\frac{1}{50}\sum_{i=1}^{50} y_{\pi(i)}$ | **Perfect operational fit.** Reviewers audit 50 URLs per week. Precision@50 measures the exact accuracy within the editor's actionable review batch. | ✅ **Primary Selection Metric** |

### Mathematical Definition of Precision@K
Let $\pi$ denote the permutation of items sorted by predicted score $\hat{s}$ in descending order:
$$\hat{s}_{\pi(1)} \ge \hat{s}_{\pi(2)} \ge \dots \ge \hat{s}_{\pi(N)}$$
$$\text{Precision@}K = \frac{1}{K} \sum_{i=1}^K y_{\pi(i)}$$

## 5. Validation Architecture: Grouped Client Holdout

Why standard random K-fold cross-validation is fatally flawed on search data:
1. **Domain Leakage:** Articles from the same client share identical domain authority, root backlink profiles, CMS technical infrastructure, and brand search elasticity.
2. **Spurious Correlation:** Random row splits allow the model to memorize client-specific baseline traffic levels rather than learning generalizable decay dynamics.

### The Grouped Validation Protocol
- Split variable: `client_id` (32 distinct clients).
- All pages from a given client belong exclusively to either Train or Test.
- This forces the ranker to prove generalization on *completely unseen client websites*.

## 6. Rubric Pass Checklist (ML-03)

- [x] **Task type formalization:** Binary risk scoring converted to descending priority queue ranking.
- [x] **Operational justification:** Fixed human reviewer capacity ($K=50$ pages/week) mandates ranking over hard binary thresholding.
- [x] **Temporal window alignment:** Trailing 90-day feature window strictly precedes forward 30-day evaluation horizon.
- [x] **Target definition:** Observed $\ge 15\%$ organic click decline; base rate measured at 54.2%.
- [x] **Leakage audit:** Explicitly banned `trend_direction`, `trend_pct`, and existing product flags from feature inputs.
- [x] **Selection metric justified:** Precision@50 chosen over Accuracy and ROC AUC due to reviewer capacity bounds.
- [x] **Split strategy justified:** Grouped client-holdout protocol prevents domain-authority leakage.